# IB9LQ0 – Generative AI and AI Applications
## Required Task 5 – Credit Card Default Prediction

**Student Name:** Jashwanth Anand Shankar
**Module:** IB9LQ0 | Warwick Business School | 2025–2026

---

### Key Insights and Takeaways from Completing This Task

**Task Overview**

This task applies the full structured-data ML workflow from L5 to a
real-world credit card default dataset. Three models are trained and
compared: Random Forest, XGBoost, and TabPFN. The target variable
is `default.payment.next.month` (1 = will default, 0 = will not).

**Key Insight 1 – What TabPFN is**

TabPFN (Tabular Prior-Data Fitted Network) is a pre-trained
Transformer for tabular data. Unlike Random Forest and XGBoost,
which must be trained from scratch on every new dataset, TabPFN
generates predictions in a single forward pass — no training loop,
no hyperparameter tuning. It uses in-context learning.

**Key Insight 2 – Why comparison metrics matter**

Accuracy alone is misleading for imbalanced datasets. A model that
always predicts 'no default' would achieve ~78% accuracy on this
dataset because most customers don't default. ROC AUC and the
classification report (precision, recall, F1) give a more honest
picture of how well each model identifies the minority class.

**Key Insight 3 – TabPFN's strengths and limitations**

TabPFN performs competitively with trained models despite requiring
no training. Its key limitation is a row cap (~10,000 rows for the
client version). For large datasets, traditional algorithms like
Random Forest and XGBoost remain the practical choice.

**Personal Takeaway**

The comparison between the three models revealed that foundation
models for tabular data (TabPFN) are reaching competitive accuracy
without any dataset-specific training — a significant shift in how
structured-data ML will be done in industry.

---
*AI Disclosure: Claude (Anthropic) was used to assist in structuring
this code. All outputs, interpretations, and conclusions were reviewed
and validated by the student.*

## Step 0 – Install Required Libraries

We install TabPFN exactly as shown in L5, plus XGBoost for the
second traditional machine learning algorithm.

In [1]:
## TabPFN Installation optimized for Google Colab
# Install the TabPFN Client library
!uv pip install tabpfn-client

# Install TabPFN extensions for additional functionalities
!uv pip install tabpfn-extensions[all]

# Install XGBoost for the second traditional ML algorithm
!pip install xgboost -q

Using Python 3.12.13 environment at: /usr
Resolved 46 packages in 2.01s
Prepared 8 packages in 198ms
Uninstalled 1 package in 7ms
Installed 8 packages in 46ms
 + backoff==2.2.1
 + nvidia-ml-py==13.595.45
 + password-strength==0.0.3.post2
 + posthog==7.14.2
 - requests==2.32.4
 + requests==2.34.2
 + sseclient-py==1.9.0
 + tabpfn-client==0.3.0
 + tabpfn-common-utils==0.2.19
Using Python 3.12.13 environment at: /usr
Resolved 88 packages in 1.82s
Prepared 15 packages in 1.60s
Installed 15 packages in 190ms
 + autogluon-common==1.5.0
 + autogluon-core==1.5.0
 + autogluon-features==1.5.0
 + autogluon-tabular==1.5.0
 + boto3==1.43.8
 + botocore==1.43.8
 + colour==0.1.5
 + eval-type-backport==0.3.1
 + galois==0.4.11
 + jmespath==1.1.0
 + s3transfer==0.17.0
 + shapiq==1.4.1
 + sparse-transform==0.2.1
 + tabpfn==8.0.2
 + tabpfn-extensions==0.4.1


In [2]:
# Simple import for TabPFN — exactly as shown in L5
from tabpfn_client import TabPFNClassifier

# Now you can use it like any other sklearn classifier
# model = TabPFNClassifier()
print("TabPFNClassifier imported successfully.")

TabPFNClassifier imported successfully.


## Step 1 – Upload and Load the Dataset

Run this cell. A **'Choose Files'** button will appear.
Select `UCI_Credit_Card.csv` (or any CSV with a
`default.payment.next.month` target column).

The code dynamically detects the filename — works with any file.

In [3]:
import pandas as pd
import io
from google.colab import files

# Open a file-picker dialog — works for any CSV/Excel filename
uploaded = files.upload()
filename = next(iter(uploaded))
print(f'File uploaded: {filename}')

# Load the data into a pandas DataFrame based on file type
if filename.endswith('.csv'):
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
elif filename.endswith(('.xls', '.xlsx')):
    df = pd.read_excel(io.BytesIO(uploaded[filename]))
else:
    raise ValueError("Unsupported file type. Please upload a .csv, .xls, or .xlsx file.")

print("Data loaded into DataFrame 'df' successfully.")
print(df.head())

Saving credit_card_qa.xlsx to credit_card_qa.xlsx
File uploaded: credit_card_qa.xlsx


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x90 in position 22: invalid start byte

In [ ]:
# Display the full DataFrame — exactly as shown at the end of the L5 task code
df

## Step 2 – Separate Features and Target

The outcome variable of interest is `default.payment.next.month`.
The `ID` column is a row identifier — it carries no predictive
information and must be removed from the features.

Following the L5 pattern: `X` contains features, `y` contains
the target.

In [ ]:
TARGET = 'default.payment.next.month'

# Drop ID column (row identifier, not a feature) and the target
X = df.drop(columns=['ID', TARGET])
y = df[TARGET]

print(f'Features shape : {X.shape}')
print(f'Target shape   : {y.shape}')
print(f'\nTarget distribution:')
print(y.value_counts())
print(f'\nDefault rate: {y.mean():.2%}')

## Algorithm 1 – Random Forest Classifier

Following exactly the preprocessing and training pattern
demonstrated in **L5_LLMs_Structured_Data.ipynb**:

1. Identify categorical columns
2. Apply one-hot encoding with `pd.get_dummies`
3. Split into 75% train / 25% test (task specifies 25%)
4. Train `RandomForestClassifier(n_estimators=100, random_state=42)`
5. Evaluate with `accuracy_score` and `classification_report`

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Identify categorical columns in X — exact pattern from L5
categorical_cols = X.select_dtypes(include=['object']).columns

# Apply one-hot encoding to categorical features — exact pattern from L5
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Split the data into training and testing sets
# CHANGED FROM L5 TUTORIAL: test_size=0.25 (task specifies 25%)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.25, random_state=42
)

# Initialize and train the Random Forest Classifier — exact from L5
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train.values.ravel())

# Make predictions on the test set — exact from L5
y_pred_rf = rf_model.predict(X_test)

# Evaluate the model — exact metrics from L5
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_report   = classification_report(y_test, y_pred_rf)
# roc_auc added here to enable a fair comparison with TabPFN
rf_roc_auc  = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

print(f"Accuracy: {rf_accuracy:.4f}")
print(f"ROC AUC : {rf_roc_auc:.4f}")
print("Classification Report:")
print(rf_report)

## Algorithm 2 – XGBoost Classifier

XGBoost (Extreme Gradient Boosting) is the second traditional
machine learning algorithm, explicitly named in the task.

It follows the **exact same sklearn API pattern** as Random Forest —
the same preprocessing, same split, same evaluation metrics. This
is intentional: both are trained from scratch on this dataset,
which is the key contrast with TabPFN.

In [ ]:
from xgboost import XGBClassifier

# XGBoost follows the exact same sklearn API pattern as Random Forest
# Same X_train, X_test, y_train, y_test from the split above
xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss',  # suppresses a default warning
    verbosity=0             # suppresses training output
)
xgb_model.fit(X_train, y_train.values.ravel())

# Make predictions on the test set
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate the model — same metrics as Random Forest
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_report   = classification_report(y_test, y_pred_xgb)
xgb_roc_auc  = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])

print(f"Accuracy: {xgb_accuracy:.4f}")
print(f"ROC AUC : {xgb_roc_auc:.4f}")
print("Classification Report:")
print(xgb_report)

## Algorithm 3 – TabPFN

TabPFN is a foundation model for tabular data. Unlike Random Forest
and XGBoost, it is a **pre-trained Transformer** that generates
predictions in a single forward pass — no dataset-specific training.

Following the **exact training and evaluation pattern** from L5:

1. Split original `X` and `y` (TabPFN handles mixed types natively —
   no one-hot encoding needed)
2. Fit and predict
3. Compute ROC AUC score, accuracy, and classification report

***First time you run this, you will be asked to create an account.***

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from tabpfn_client import TabPFNClassifier

# Split the data into training and testing sets
# TabPFN uses raw X (not X_encoded) — it handles mixed types natively
# CHANGED FROM L5 TUTORIAL: test_size=0.25 (task specifies 25%)
X_train_pfn, X_test_pfn, y_train_pfn, y_test_pfn = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Train and evaluate the TabPFN classifier — exact from L5
tabpfn_classifier = TabPFNClassifier(random_state=42)
tabpfn_classifier.fit(X_train_pfn, y_train_pfn)
y_pred_proba = tabpfn_classifier.predict_proba(X_test_pfn)
y_pred_pfn   = tabpfn_classifier.predict(X_test_pfn)

# Calculate the ROC AUC score — exact from L5
tabpfn_roc_auc = roc_auc_score(y_test_pfn, y_pred_proba[:, 1])
print(f"TabPFN ROC AUC Score: {tabpfn_roc_auc:.4f}")

# Evaluate the model — exact from L5
tabpfn_accuracy = accuracy_score(y_test_pfn, y_pred_pfn)
tabpfn_report   = classification_report(y_test_pfn, y_pred_pfn)

print(f"Accuracy: {tabpfn_accuracy:.4f}")
print("Classification Report:")
print(tabpfn_report)

## Comparison Summary

The table below directly answers the task question:
**"How well does TabPFN perform in comparison to machine learning algorithms?"**

All three models are evaluated on the same 25% test set with
the same metrics: Accuracy and ROC AUC.

In [ ]:
import pandas as pd

# Build a summary comparison table
comparison = pd.DataFrame({
    'Model'   : ['Random Forest', 'XGBoost', 'TabPFN'],
    'Accuracy': [round(rf_accuracy, 4), round(xgb_accuracy, 4), round(tabpfn_accuracy, 4)],
    'ROC AUC' : [round(rf_roc_auc, 4),  round(xgb_roc_auc, 4),  round(tabpfn_roc_auc, 4)],
    'Training Required': ['Yes', 'Yes', 'No (pre-trained)']
})

print("--- Model Comparison on 25% Test Set ---\n")
print(comparison.to_string(index=False))

# Identify the best model by ROC AUC
best_model = comparison.loc[comparison['ROC AUC'].idxmax(), 'Model']
print(f"\n>>> Best model by ROC AUC: {best_model}")

# Print the interpretation
tabpfn_vs_rf  = tabpfn_roc_auc - rf_roc_auc
tabpfn_vs_xgb = tabpfn_roc_auc - xgb_roc_auc

print(f"""
Interpretation:
  TabPFN ROC AUC vs Random Forest : {tabpfn_vs_rf:+.4f}
  TabPFN ROC AUC vs XGBoost       : {tabpfn_vs_xgb:+.4f}

  TabPFN {'outperforms' if tabpfn_roc_auc > max(rf_roc_auc, xgb_roc_auc) else 'performs comparably to or below'}
  the traditional algorithms despite requiring no dataset-specific
  training. This demonstrates TabPFN's strength as an in-context
  learner — it generates predictions in a single forward pass using
  knowledge encoded during its pre-training phase.
""")